# Xóa vật thừa trong ảnh — Hybrid Mask2Former (panoptic) + YOLOv8-seg, IS-Net, LaMa (Colab Free / T4)

**Pipeline:** ảnh →
1. **Mask2Former ADE20K** chia *toàn bộ* ảnh thành vùng (150 lớp: người, xe, cột, biển hiệu, thùng rác... và nền: trời, cây, nhà, đường...)
2. **YOLOv8-seg** (80 lớp COCO) chạy song song — bổ sung vật panoptic bỏ sót hoặc gộp mất (túi xách, điện thoại đang cầm...). Trùng nhau thì **dùng mask panoptic**.
3. **Bộ dò dây điện** (thử nghiệm)
4. **IS-Net (BRIA RMBG-1.4)** đánh dấu vùng chủ thể: vật to nhất trong vùng (+ vật cùng lớp cỡ tương đương, vd. cặp đôi) = chủ thể. **Vật khác dính chủ thể vẫn được liệt kê**, mặc định không tick.
5. **Chấm điểm từng vật** → **LaMa** xóa & lấp nền.

**Cách dùng:** `Runtime → Change runtime type → T4 GPU`, rồi chạy lần lượt từ trên xuống.

| Màu | Ý nghĩa |
|---|---|
| 🟩 xanh lá | chủ thể (IS-Net) — không xóa |
| 🟦 xanh dương | dính chủ thể (túi, điện thoại...) — liệt kê, mặc định không xóa; muốn xóa thì thêm id vào `ADD_IDS` |
| 🟥 đỏ | đề xuất xóa |
| 🟨 vàng | ứng viên điểm thấp — giữ |
| (không tô) | nền: trời, cây, nhà, đường... — không bao giờ xóa |

**Giới hạn cần biết**
- **Chủ thể do IS-Net đánh dấu**. Nếu IS-Net không thấy chủ thể nào, ảnh coi như không có chủ thể và *mọi* vật (kể cả người chính) có thể bị đề xuất xóa — luôn xem lại trước khi xóa, hoặc chỉ định tay bằng `SUBJECT_IDS`.
- ADE20K không có lớp **dây điện** → dò riêng bằng heuristic, có thể sót/nhầm.
- "Vật thừa" quyết định bằng **luật** (ô Cấu hình), chưa phải model học.
- ⚠ **Giấy phép BRIA RMBG-1.4: chỉ phi thương mại.** Thương mại hóa cần mua license BRIA hoặc đổi sang `isnet-general-use`.

## 1. Kiểm tra GPU & cài thư viện

In [4]:
!nvidia-smi -L

GPU 0: NVIDIA GeForce RTX 3050 Laptop GPU (UUID: GPU-1ba68309-9250-fa54-2075-35582ec24389)


In [5]:
%pip install -q ultralytics onnxruntime-gpu

Note: you may need to restart the kernel to use updated packages.


## 2. Cấu hình

In [6]:
# ==== CẤU HÌNH — chỉnh ở đây ====

# --- Panoptic segmentation: Mask2Former Swin-L pretrain ADE20K (150 lớp: vật + nền) ---
PANOPTIC_MODEL      = "facebook/mask2former-swin-large-ade-panoptic"
PANOPTIC_THRESHOLD  = 0.5     # ngưỡng tin cậy cho mỗi vùng; hạ xuống (0.3) nếu bị sót vật nhỏ
PANOPTIC_FP16       = True    # chạy nửa độ chính xác trên GPU cho nhẹ VRAM T4; nếu kết quả lạ -> False

# --- YOLOv8-seg (COCO 80 lớp) — chạy song song với panoptic ---
YOLO_WEIGHTS = "yolov8x-seg.pt"   # n/s/m/l/x; T4 chạy x được
YOLO_IMGSZ   = 1280               # ảnh to, vật nhỏ ở xa -> 1280; muốn nhanh -> 640
YOLO_CONF    = 0.25
DEDUP_IOU    = 0.5                # vật YOLO trùng vùng panoptic (IoU >= 0.5) -> bỏ bản YOLO, dùng mask panoptic

# --- IS-Net (BRIA RMBG-1.4, bản ONNX) — QUYẾT ĐỊNH CHỦ THỂ ---
# ⚠ Giấy phép RMBG-1.4: chỉ dùng phi thương mại; thương mại phải mua license của BRIA.
ISNET_REPO      = "briaai/RMBG-1.4"
ISNET_FILE      = "onnx/model.onnx"
ISNET_SIZE      = 1024
ISNET_THRESHOLD = 0.5    # xác suất >= ngưỡng -> pixel thuộc chủ thể
SUBJECT_OVERLAP = 0.5    # vật có >= 50% diện tích nằm trong mask IS-Net -> nằm trong vùng chủ thể
GROUP_AREA_RATIO = 0.4   # trong vùng IS-Net: vật to nhất là chủ thể; vật CÙNG LỚP to >= 40% nó cũng là chủ thể (ảnh cặp/nhóm)
# Các vật khác trong/dính vùng chủ thể (túi, điện thoại, người chồng lên...) -> vẫn LIỆT KÊ nhưng mặc định không tick xóa
# IS-Net không ra vùng chủ thể nào -> ảnh coi như không có chủ thể

# --- Lớp NỀN: không bao giờ đề xuất xóa ---
BACKGROUND_LABELS = {
    "wall", "building", "sky", "floor", "tree", "ceiling", "road", "grass", "sidewalk", "earth",
    "mountain", "water", "sea", "field", "sand", "river", "hill", "land", "lake", "path", "rock",
    "skyscraper", "house", "bridge", "stairs", "stairway", "step", "dirt track", "runway", "waterfall",
    "windowpane", "door", "railing", "fence", "column", "countertop", "palm", "plant", "flower",
    "swimming pool", "pier", "tower", "hovel", "canopy", "awning",
}

# --- Chấm điểm "vật thừa" theo lớp (tên ADE20K: phần trước dấu phẩy, chữ thường; tên COCO: như YOLO) ---
# Lớp không có trong bảng (và không phải nền) dùng DEFAULT_PRIOR. Ô "Tải model" in ra toàn bộ 150 tên lớp để bạn chỉnh.
DEFAULT_PRIOR = 0.30
CLASS_PRIOR = {
    "person": 0.55,
    "car": 0.45, "van": 0.45, "truck": 0.50, "bus": 0.45, "bicycle": 0.45, "minibike": 0.50,
    "boat": 0.25, "ship": 0.20, "airplane": 0.30,
    "pole": 0.65, "streetlight": 0.60, "traffic light": 0.60, "signboard": 0.60, "trade name": 0.55,
    "poster": 0.50, "bulletin board": 0.50, "flag": 0.40, "booth": 0.30,
    "ashcan": 0.65, "barrel": 0.45, "box": 0.45, "bag": 0.50, "basket": 0.40,
    "bottle": 0.55, "glass": 0.45, "plate": 0.30, "tray": 0.35, "pot": 0.30, "food": 0.20,
    "towel": 0.40, "apparel": 0.45, "plaything": 0.40, "ball": 0.30, "blanket": 0.30,
    "fan": 0.30, "radiator": 0.35, "sconce": 0.20, "light": 0.20, "lamp": 0.20,
    "crt screen": 0.35, "monitor": 0.35, "screen": 0.35, "computer": 0.35, "television receiver": 0.35,
    "clock": 0.25, "vase": 0.20, "sculpture": 0.15, "fountain": 0.10, "tent": 0.35,
    "stool": 0.30, "chair": 0.30, "swivel chair": 0.30, "seat": 0.25, "bench": 0.30,
    "cushion": 0.25, "pillow": 0.25, "rug": 0.15, "animal": 0.20,
    # tên lớp COCO (YOLO) chưa có ở trên
    "motorcycle": 0.50, "train": 0.30, "fire hydrant": 0.55, "stop sign": 0.60, "parking meter": 0.60,
    "backpack": 0.50, "handbag": 0.50, "suitcase": 0.45, "umbrella": 0.35, "tie": 0.10,
    "cup": 0.50, "wine glass": 0.35, "fork": 0.30, "knife": 0.30, "spoon": 0.30, "bowl": 0.25,
    "couch": 0.15, "potted plant": 0.15, "dining table": 0.10, "toilet": 0.40, "tv": 0.35, "laptop": 0.35,
    "mouse": 0.45, "remote": 0.50, "keyboard": 0.40, "cell phone": 0.50, "book": 0.30, "scissors": 0.40,
    "teddy bear": 0.30, "hair drier": 0.50, "toothbrush": 0.50, "kite": 0.40, "frisbee": 0.35,
    "sports ball": 0.30, "skateboard": 0.35, "surfboard": 0.30, "bird": 0.20, "cat": 0.20, "dog": 0.20,
}
SUGGEST_THRESHOLD  = 0.50   # điểm >= ngưỡng -> đề xuất xóa
EDGE_MARGIN        = 0.02   # vật chạm mép ảnh (trong 2%) -> cộng điểm
MAX_REMOVE_AREA    = 0.15   # vật chiếm > 15% ảnh -> xóa dễ lỗi, trừ điểm
BELONGS_TO_SUBJECT = 0.30   # >= 30% diện tích nằm trong vùng chủ thể (nới rộng) -> "dính chủ thể": liệt kê, không tick

# --- Dò dây điện (thử nghiệm) — ADE20K không có lớp dây điện ---
DETECT_WIRES   = True
WIRE_SCORE     = 0.70
WIRE_MIN_LEN   = 0.20
WIRE_BH_THRESH = 18

# --- Xóa & lấp nền (LaMa) ---
LAMA_URL       = "https://github.com/enesmsahin/simple-lama-inpainting/releases/download/v0.1.0/big-lama.pt"
LAMA_PATH      = "/content/big-lama.pt"
LAMA_MAX_SIDE  = 1024
MASK_DILATE    = 0.006
CROP_MARGIN    = 1.0

OUTPUT_DIR = "/content/output"

## 3. Tải model

In [7]:
# ==== TẢI MODEL (chạy 1 lần) ====
import os
import cv2
import numpy as np
import torch
import onnxruntime as ort
from huggingface_hub import hf_hub_download
from ultralytics import YOLO
from transformers import AutoImageProcessor, Mask2FormerForUniversalSegmentation

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Thiết bị:", DEVICE)

# --- YOLOv8-seg ---
yolo = YOLO(YOLO_WEIGHTS)
print("YOLO:", YOLO_WEIGHTS, "-", len(yolo.names), "lớp")

# --- Panoptic: Mask2Former ADE20K ---
pan_processor = AutoImageProcessor.from_pretrained(PANOPTIC_MODEL)
pan_model = Mask2FormerForUniversalSegmentation.from_pretrained(PANOPTIC_MODEL).to(DEVICE).eval()
if PANOPTIC_FP16 and DEVICE == "cuda":
    pan_model = pan_model.half()


def norm_label(s):
    """'signboard, sign' -> 'signboard'"""
    return s.split(",")[0].strip().lower()

PAN_LABELS = {int(k): norm_label(v) for k, v in pan_model.config.id2label.items()}
print(f"Panoptic: {len(PAN_LABELS)} lớp ->", ", ".join(PAN_LABELS[k] for k in sorted(PAN_LABELS)))


# --- IS-Net: BRIA RMBG-1.4 (ONNX) ---
class ISNet:
    """Trả về bản đồ xác suất chủ thể (H x W, 0..1)."""
    def __init__(self, path):
        self.sess = ort.InferenceSession(path, providers=["CUDAExecutionProvider", "CPUExecutionProvider"])
        self.inp = self.sess.get_inputs()[0].name
        print("IS-Net chạy trên:", self.sess.get_providers()[0])

    def __call__(self, img_bgr):
        H, W = img_bgr.shape[:2]
        rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        x = cv2.resize(rgb, (ISNET_SIZE, ISNET_SIZE), interpolation=cv2.INTER_LINEAR).astype(np.float32) / 255.0
        x = (x - 0.5).transpose(2, 0, 1)[None]          # chuẩn hoá giống model card: mean 0.5, std 1.0
        out = self.sess.run(None, {self.inp: x})[0]
        p = out.reshape(out.shape[-2], out.shape[-1]).astype(np.float32)
        if p.min() < 0 or p.max() > 1:                   # phòng trường hợp bản ONNX chưa có sigmoid
            p = 1 / (1 + np.exp(-p))
        return cv2.resize(p, (W, H), interpolation=cv2.INTER_LINEAR)

isnet = ISNet(hf_hub_download(ISNET_REPO, ISNET_FILE))


# --- LaMa ---
class LamaInpainter:
    """LaMa (big-lama, TorchScript). Vào: ảnh RGB uint8 + mask uint8 (255 = xóa). Ra: ảnh RGB uint8."""
    def __init__(self, path, device):
        self.device = device
        self.model = torch.jit.load(path, map_location=device).eval()

    @torch.inference_mode()
    def __call__(self, rgb, mask):
        h, w = rgb.shape[:2]
        ph, pw = (8 - h % 8) % 8, (8 - w % 8) % 8          # LaMa cần kích thước chia hết cho 8
        rgb_p = np.pad(rgb, ((0, ph), (0, pw), (0, 0)), mode="reflect")
        m_p = np.pad(mask, ((0, ph), (0, pw)), mode="reflect")
        img = torch.from_numpy(rgb_p).permute(2, 0, 1)[None].float().div(255).to(self.device)
        m = torch.from_numpy((m_p > 127).astype(np.float32))[None, None].to(self.device)
        out = self.model(img, m)[0].permute(1, 2, 0).clamp(0, 1).mul(255).byte().cpu().numpy()
        return out[:h, :w]


if not os.path.exists(LAMA_PATH):
    torch.hub.download_url_to_file(LAMA_URL, LAMA_PATH)
lama = LamaInpainter(LAMA_PATH, DEVICE)
print("LaMa: sẵn sàng")
os.makedirs(OUTPUT_DIR, exist_ok=True)

d:\Anaconda\envs\banghe\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Thiết bị: cpu
YOLO: yolov8x-seg.pt - 80 lớp


ImportError: 
Mask2FormerLoss requires the scipy library but it was not found in your environment. You can install it with pip:
`pip install scipy`. Please note that you may need to restart your runtime after installation.


## 4. Code xử lý (chạy để nạp hàm)

In [ ]:
# ==== LOGIC CHÍNH: panoptic + YOLOv8 -> IS-Net đánh dấu chủ thể -> chấm điểm vật thừa ====
import cv2
import numpy as np
import torch


def _diag(h, w):
    return float(np.hypot(h, w))


def _sharpness(gray, mask):
    """Độ nét = phương sai Laplacian bên trong mask (co mask lại để bỏ viền)."""
    lap = cv2.Laplacian(gray, cv2.CV_64F)
    m = cv2.erode(mask.astype(np.uint8), np.ones((5, 5), np.uint8)) > 0
    if m.sum() < 50:
        m = mask
    if m.sum() == 0:
        return 0.0
    return float(lap[m].var())


def _bbox(m):
    ys, xs = np.where(m)
    return int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max())


@torch.inference_mode()
def segment(img_bgr):
    """Panoptic: mọi pixel thuộc 1 vùng (vật hoặc nền). Trả về list object: name, conf, mask, bbox, area, is_bg."""
    H, W = img_bgr.shape[:2]
    rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    inputs = pan_processor(images=rgb, return_tensors="pt").to(DEVICE)
    if PANOPTIC_FP16 and DEVICE == "cuda":
        inputs["pixel_values"] = inputs["pixel_values"].half()
    outputs = pan_model(**inputs)
    if PANOPTIC_FP16 and DEVICE == "cuda":   # hậu xử lý ở fp32 cho ổn định
        outputs.class_queries_logits = outputs.class_queries_logits.float()
        outputs.masks_queries_logits = outputs.masks_queries_logits.float()
    res = pan_processor.post_process_panoptic_segmentation(
        outputs, threshold=PANOPTIC_THRESHOLD, target_sizes=[(H, W)])[0]
    seg = res["segmentation"].cpu().numpy()
    objs = []
    for info in res["segments_info"]:
        m = seg == info["id"]
        if m.sum() == 0:
            continue
        name = PAN_LABELS[int(info["label_id"])]
        objs.append(dict(name=name, conf=float(info.get("score", 1.0)), mask=m, bbox=_bbox(m),
                         area=float(m.sum()) / (H * W), source="panoptic", is_bg=name in BACKGROUND_LABELS))
    return objs


def segment_yolo(img_bgr):
    """YOLOv8-seg: các vật thuộc 80 lớp COCO."""
    H, W = img_bgr.shape[:2]
    res = yolo.predict(img_bgr, imgsz=YOLO_IMGSZ, conf=YOLO_CONF, retina_masks=True, verbose=False)[0]
    objs = []
    if res.masks is None:
        return objs
    masks = res.masks.data.cpu().numpy()
    clss = res.boxes.cls.cpu().numpy().astype(int)
    confs = res.boxes.conf.cpu().numpy()
    for i in range(len(masks)):
        m = masks[i]
        if m.shape != (H, W):
            m = cv2.resize(m.astype(np.float32), (W, H), interpolation=cv2.INTER_LINEAR)
        m = m > 0.5
        if m.sum() == 0:
            continue
        objs.append(dict(name=res.names[int(clss[i])], conf=float(confs[i]), mask=m, bbox=_bbox(m),
                         area=float(m.sum()) / (H * W), source="yolo", is_bg=False))
    return objs


def merge_segments(pan_objs, yolo_objs):
    """Ưu tiên panoptic: vật YOLO trùng (IoU >= DEDUP_IOU) với vùng panoptic không phải nền -> bỏ bản YOLO.
    Vật YOLO còn lại (panoptic bỏ sót / lẫn vào vùng khác, vd. túi xách dính vào người) được thêm vào."""
    things = [o for o in pan_objs if not o["is_bg"]]
    out = list(pan_objs)
    for y in yolo_objs:
        dup = False
        for p in things:
            inter = (y["mask"] & p["mask"]).sum()
            if inter and inter / (y["mask"] | p["mask"]).sum() >= DEDUP_IOU:
                dup = True
                p.setdefault("also_yolo", y["name"])
                break
        if not dup:
            out.append(y)
    return out


def detect_wires(img_bgr, exclude_mask=None):
    """Dò dây điện/dây cáp: nét mảnh, tối hơn nền, dài, hai bên có màu giống nhau (thường là trời).
    Trả về mask bool hoặc None. Đây là heuristic, có thể sót hoặc nhận nhầm."""
    H, W = img_bgr.shape[:2]
    s = min(1.0, 1024.0 / max(H, W))
    small = cv2.resize(img_bgr, (int(W * s), int(H * s)), interpolation=cv2.INTER_AREA) if s < 1 else img_bgr.copy()
    h, w = small.shape[:2]
    gray = cv2.cvtColor(small, cv2.COLOR_BGR2GRAY)
    gray = cv2.GaussianBlur(gray, (3, 3), 0)
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9))
    bh = cv2.morphologyEx(gray, cv2.MORPH_BLACKHAT, k)          # làm nổi nét mảnh tối trên nền sáng
    bw = (bh > WIRE_BH_THRESH).astype(np.uint8) * 255
    min_len = int(WIRE_MIN_LEN * max(h, w))
    lines = cv2.HoughLinesP(bw, 1, np.pi / 180, threshold=60, minLineLength=min_len, maxLineGap=int(0.02 * max(h, w)))
    if lines is None:
        return None

    lab = cv2.cvtColor(small, cv2.COLOR_BGR2LAB).astype(np.float32)
    out = np.zeros((h, w), np.uint8)
    found = 0
    for x1, y1, x2, y2 in np.asarray(lines).reshape(-1, 4):
        L = np.hypot(x2 - x1, y2 - y1)
        if L < 1:
            continue
        nx, ny = -(y2 - y1) / L, (x2 - x1) / L                    # pháp tuyến
        ok, tot = 0, 0
        for t in np.linspace(0.05, 0.95, 25):
            cx, cy = x1 + t * (x2 - x1), y1 + t * (y2 - y1)
            pts = []
            for d in (-6, 0, 6):
                px, py = int(round(cx + d * nx)), int(round(cy + d * ny))
                if not (0 <= px < w and 0 <= py < h):
                    break
                pts.append(lab[py, px])
            if len(pts) < 3:
                continue
            tot += 1
            a, c, b = pts
            sides_same = np.abs(a - b).mean() < 12                  # hai bên dây cùng nền
            darker = (a[0] + b[0]) / 2 - c[0] > 6                   # dây tối hơn nền
            if sides_same and darker:
                ok += 1
        if tot >= 10 and ok / tot >= 0.6:
            cv2.line(out, (int(x1), int(y1)), (int(x2), int(y2)), 255, 3)
            found += 1
    if found == 0:
        return None
    m = cv2.resize(out, (W, H), interpolation=cv2.INTER_LINEAR) > 0 if s < 1 else out > 0
    if exclude_mask is not None:
        m &= ~exclude_mask
    return m if m.sum() > 0 else None


def find_subjects(objs, img_bgr, prob=None):
    """IS-Net đánh dấu vùng chủ thể. Trong vùng đó (vật có >= SUBJECT_OVERLAP diện tích nằm trong mask IS-Net):
    vật to nhất = chủ thể; vật cùng lớp to >= GROUP_AREA_RATIO lần nó cũng là chủ thể.
    Trả về (list index chủ thể, mask IS-Net)."""
    if prob is None:
        prob = isnet(img_bgr)
    fg = prob >= ISNET_THRESHOLD
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    for o in objs:
        o["sharp"] = _sharpness(gray, o["mask"])
        o["in_fg"] = float((o["mask"] & fg).sum()) / max(1, o["mask"].sum())
    max_sharp = max((o["sharp"] for o in objs), default=1.0) or 1.0
    for o in objs:
        o["sharp_n"] = o["sharp"] / max_sharp
    if not fg.any():
        return [], fg
    inside = [i for i, o in enumerate(objs) if not o["is_bg"] and o["in_fg"] >= SUBJECT_OVERLAP]
    if not inside:
        return [], fg
    main = max(inside, key=lambda i: objs[i]["area"])
    subs = [i for i in inside if i == main or (objs[i]["name"] == objs[main]["name"]
                                              and objs[i]["area"] >= GROUP_AREA_RATIO * objs[main]["area"])]
    return subs, fg


def score_distractors(objs, subj_idx, img_bgr, fg=None):
    """Gán role/score/suggest/reasons cho từng vùng.
    role: subject | attached (dính chủ thể: liệt kê, không tick) | candidate | background"""
    H, W = img_bgr.shape[:2]
    core = np.zeros((H, W), bool)                 # chỉ các vật chủ thể (để đo "dính")
    for i in subj_idx:
        core |= objs[i]["mask"]
    subj_mask = core | (fg if fg is not None else False)
    has_subj = bool(subj_mask.any())
    d = max(3, int(0.02 * _diag(H, W)))
    subj_dil = cv2.dilate(subj_mask.astype(np.uint8), np.ones((d, d), np.uint8)) > 0
    subj_area = float(subj_mask.sum()) / (H * W)
    subj_sharp = max((objs[i]["sharp_n"] for i in subj_idx), default=1.0)
    mx, my = EDGE_MARGIN * W, EDGE_MARGIN * H

    for i, o in enumerate(objs):
        o["id"] = i
        if i in subj_idx:
            o.update(role="subject", score=0.0, suggest=False,
                     reasons=[f"chủ thể (IS-Net phủ {o.get('in_fg', 1):.0%})"])
            continue
        if o.get("source") == "wire":
            o.update(role="candidate", score=WIRE_SCORE, suggest=WIRE_SCORE >= SUGGEST_THRESHOLD,
                     reasons=["dây điện/dây cáp (thử nghiệm)"])
            continue
        if o.get("is_bg"):
            o.update(role="background", score=0.0, suggest=False, reasons=["nền"])
            continue

        inside = (o["mask"] & subj_dil).sum() / max(1, o["mask"].sum())
        score = CLASS_PRIOR.get(o["name"], DEFAULT_PRIOR)
        reasons = [f"lớp '{o['name']}' gốc {score:.2f}"]
        x1, y1, x2, y2 = o["bbox"]
        if x1 <= mx or y1 <= my or x2 >= W - 1 - mx or y2 >= H - 1 - my:
            score += 0.15; reasons.append("bị cắt ở mép ảnh +0.15")
        if has_subj:
            if o["area"] < 0.25 * subj_area:
                score += 0.15; reasons.append("nhỏ hơn nhiều so với chủ thể +0.15")
            if subj_idx and o.get("sharp_n", 1) < 0.5 * subj_sharp:
                score += 0.10; reasons.append("mờ hơn chủ thể (hậu cảnh) +0.10")
        elif o["area"] < 0.02:
            score += 0.10; reasons.append("vật nhỏ, ảnh không có chủ thể +0.10")
        if o["area"] > MAX_REMOVE_AREA:
            score -= 0.30; reasons.append(f"quá lớn ({o['area']:.0%} ảnh), xóa dễ lỗi -0.30")
        score = float(np.clip(score, 0, 1))

        if has_subj and (o.get("in_fg", 0) >= SUBJECT_OVERLAP or inside >= BELONGS_TO_SUBJECT):
            reasons.insert(0, f"DÍNH CHỦ THỂ ({max(inside, o.get('in_fg', 0)):.0%}) — mặc định không xóa")
            o.update(role="attached", score=score, suggest=False, reasons=reasons)
            continue
        if inside > 0:
            score = max(0.0, score - 0.10); reasons.append("chạm chủ thể -0.10")
        o.update(role="candidate", score=score, suggest=score >= SUGGEST_THRESHOLD, reasons=reasons)
    return objs


def analyze(img_bgr, subject_ids=None):
    """Toàn bộ bước phân tích. subject_ids: tự chỉ định chủ thể (list id) nếu chọn sai."""
    objs = merge_segments(segment(img_bgr), segment_yolo(img_bgr))
    subj, fg = find_subjects(objs, img_bgr)
    if subject_ids is not None:        # người dùng chỉ định tay -> bỏ mask IS-Net
        subj, fg = list(subject_ids), None
    if DETECT_WIRES:
        excl = np.zeros(img_bgr.shape[:2], bool) if fg is None else fg.copy()
        for i in subj:
            excl |= objs[i]["mask"]
        wm = detect_wires(img_bgr, excl)
        if wm is not None:
            objs.append(dict(name="power line", conf=1.0, mask=wm, source="wire", is_bg=False,
                             bbox=_bbox(wm), area=float(wm.sum()) / wm.size, sharp_n=1.0))
    return score_distractors(objs, subj, img_bgr, fg)

In [ ]:
# ==== HIỂN THỊ ====
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

COLORS = {  # BGR
    "subject": (60, 200, 60),          # xanh lá  = chủ thể
    "attached": (200, 160, 40),        # xanh dương = dính chủ thể (liệt kê, không tick)
    "suggest": (40, 40, 230),          # đỏ       = đề xuất xóa
    "keep": (40, 200, 230),            # vàng     = giữ lại
}


def _color(o):
    if o["role"] == "subject":
        return COLORS["subject"]
    if o["role"] == "attached" and not o["suggest"]:
        return COLORS["attached"]
    return COLORS["suggest"] if o["suggest"] else COLORS["keep"]


def draw_overlay(img_bgr, objs, selected_ids=None):
    """Tô màu từng vật + ghi số id. selected_ids: nếu truyền vào thì tô đỏ theo danh sách này."""
    vis = img_bgr.copy()
    over = img_bgr.copy()
    H, W = img_bgr.shape[:2]
    th = max(2, int(0.002 * np.hypot(H, W)))
    fs = max(0.5, 0.0008 * np.hypot(H, W))
    objs = [o for o in objs if o["role"] != "background"]   # nền không tô để ảnh đỡ rối
    for o in objs:
        c = _color(o)
        if selected_ids is not None and o["role"] in ("candidate", "attached"):
            c = COLORS["suggest"] if o["id"] in selected_ids else (COLORS["attached"] if o["role"] == "attached" else COLORS["keep"])
        over[o["mask"]] = c
        cnts, _ = cv2.findContours(o["mask"].astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(vis, cnts, -1, c, th)
    vis = cv2.addWeighted(over, 0.35, vis, 0.65, 0)
    for o in objs:
        x1, y1, _, _ = o["bbox"]
        txt = f"#{o['id']} {o['name']}"
        (tw, tht), _ = cv2.getTextSize(txt, cv2.FONT_HERSHEY_SIMPLEX, fs, th)
        y = max(tht + 4, y1)
        cv2.rectangle(vis, (x1, y - tht - 4), (x1 + tw + 4, y + 2), (0, 0, 0), -1)
        cv2.putText(vis, txt, (x1 + 2, y - 2), cv2.FONT_HERSHEY_SIMPLEX, fs, (255, 255, 255), th, cv2.LINE_AA)
    return vis


def objs_table(objs):
    role_vi = {"subject": "chủ thể", "attached": "dính chủ thể", "candidate": "ứng viên", "background": "nền"}
    rows = [dict(id=o["id"], vật=o["name"], nguồn=o.get("source", ""), vai_trò=role_vi[o["role"]],
                 điểm=round(o["score"], 2), đề_xuất_xóa="✅" if o["suggest"] else "",
                 diện_tích=f"{o['area']:.1%}", lý_do="; ".join(o["reasons"])) for o in objs]
    return pd.DataFrame(rows)


def show(*imgs_bgr, titles=None, size=7):
    n = len(imgs_bgr)
    plt.figure(figsize=(size * n, size))
    for k, im in enumerate(imgs_bgr):
        plt.subplot(1, n, k + 1)
        plt.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
        plt.axis("off")
        if titles:
            plt.title(titles[k])
    plt.tight_layout()
    plt.show()

In [ ]:
# ==== XÓA & LẤP NỀN ====
import cv2
import numpy as np


def _crop_boxes(mask, margin_ratio):
    """Mỗi cụm mask liền nhau -> một hộp cắt (có thêm ngữ cảnh xung quanh). Gộp các hộp chồng nhau."""
    H, W = mask.shape
    n, _, stats, _ = cv2.connectedComponentsWithStats(mask.astype(np.uint8), 8)
    boxes = []
    for k in range(1, n):
        x, y, w, h, a = stats[k]
        mx, my = int(max(48, w * margin_ratio)), int(max(48, h * margin_ratio))
        boxes.append([max(0, x - mx), max(0, y - my), min(W, x + w + mx), min(H, y + h + my)])
    merged = True
    while merged:                      # gộp hộp chồng nhau
        merged = False
        for i in range(len(boxes)):
            for j in range(i + 1, len(boxes)):
                a, b = boxes[i], boxes[j]
                if a[0] < b[2] and b[0] < a[2] and a[1] < b[3] and b[1] < a[3]:
                    boxes[i] = [min(a[0], b[0]), min(a[1], b[1]), max(a[2], b[2]), max(a[3], b[3])]
                    boxes.pop(j); merged = True; break
            if merged:
                break
    return boxes


def remove_objects(img_bgr, objs, ids, inpaint_fn=None):
    """Xóa các object có id trong `ids`. inpaint_fn(rgb_uint8, mask_uint8) -> rgb_uint8 (mặc định: LaMa)."""
    inpaint_fn = inpaint_fn or lama
    H, W = img_bgr.shape[:2]
    ids = set(ids)
    mask = np.zeros((H, W), bool)
    for o in objs:
        if o["id"] in ids:
            mask |= o["mask"]
    if not mask.any():
        return img_bgr.copy(), mask
    d = max(3, int(MASK_DILATE * np.hypot(H, W))) | 1
    mask = cv2.dilate(mask.astype(np.uint8), cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (d, d))) > 0

    out = img_bgr.copy()
    for x1, y1, x2, y2 in _crop_boxes(mask, CROP_MARGIN):
        crop = cv2.cvtColor(out[y1:y2, x1:x2], cv2.COLOR_BGR2RGB)
        cm = mask[y1:y2, x1:x2].astype(np.uint8) * 255
        ch, cw = crop.shape[:2]
        s = min(1.0, LAMA_MAX_SIDE / max(ch, cw))
        if s < 1:   # thu nhỏ để không hết VRAM; mask thu bằng LINEAR rồi >0 để giữ nét mảnh (dây điện)
            crop_s = cv2.resize(crop, (int(cw * s), int(ch * s)), interpolation=cv2.INTER_AREA)
            cm_s = (cv2.resize(cm, (int(cw * s), int(ch * s)), interpolation=cv2.INTER_LINEAR) > 0).astype(np.uint8) * 255
        else:
            crop_s, cm_s = crop, cm
        filled = inpaint_fn(crop_s, cm_s)
        if s < 1:
            filled = cv2.resize(filled, (cw, ch), interpolation=cv2.INTER_CUBIC)
        filled = cv2.cvtColor(filled, cv2.COLOR_RGB2BGR)
        # trộn mềm ở viền để không lộ vết cắt
        m = cm.astype(np.float32) / 255
        alpha = np.maximum(m, cv2.GaussianBlur(m, (0, 0), 2))[..., None]
        out[y1:y2, x1:x2] = (alpha * filled + (1 - alpha) * out[y1:y2, x1:x2]).astype(np.uint8)
    return out, mask

## 5. Dùng thử với 1 ảnh

In [ ]:
# ==== TẢI ẢNH LÊN ====
# Cách 1: bấm chọn file từ máy. Cách 2: đặt IMAGE_PATH = "/content/drive/MyDrive/....jpg"
IMAGE_PATH = None

if IMAGE_PATH is None:
    from google.colab import files
    up = files.upload()
    IMAGE_PATH = "/content/" + next(iter(up))

img = cv2.imread(IMAGE_PATH)
assert img is not None, f"Không đọc được ảnh: {IMAGE_PATH}"
print(IMAGE_PATH, img.shape)

In [ ]:
# ==== PHÂN TÍCH & ĐỀ XUẤT ====
# Chủ thể do IS-Net quyết định. Nếu sai: đặt SUBJECT_IDS = [id, ...] (xem cột id trong bảng) rồi chạy lại ô này.
SUBJECT_IDS = None

objs = analyze(img, subject_ids=SUBJECT_IDS)
fg_vis = (isnet(img) * 255).astype("uint8")
show(img, cv2.cvtColor(fg_vis, cv2.COLOR_GRAY2BGR), draw_overlay(img, objs), size=6,
     titles=["Ảnh gốc", "IS-Net: chủ thể (trắng)", "Xanh lá = chủ thể | Đỏ = xóa | Vàng = giữ | Xanh dương = dính chủ thể (không tick)"])
display(objs_table(objs))
print("Đề xuất xóa:", [o["id"] for o in objs if o["suggest"]])
print("Dính chủ thể (muốn xóa thì thêm vào ADD_IDS ở ô dưới):", [o["id"] for o in objs if o["role"] == "attached"])

In [ ]:
# ==== XÓA ====
# Mặc định xóa theo đề xuất. Sửa tay nếu muốn:
ADD_IDS    = []   # thêm id muốn xóa
REMOVE_IDS = []   # bỏ id khỏi danh sách xóa

ids = ({o["id"] for o in objs if o["suggest"]} | set(ADD_IDS)) - set(REMOVE_IDS)
print("Sẽ xóa:", sorted(ids))

result, used_mask = remove_objects(img, objs, ids)
show(draw_overlay(img, objs, selected_ids=ids), result, titles=["Vùng xóa (đỏ)", "Kết quả"])

import os
out_path = os.path.join(OUTPUT_DIR, os.path.splitext(os.path.basename(IMAGE_PATH))[0] + "_clean.png")
cv2.imwrite(out_path, result)
print("Đã lưu:", out_path)
# from google.colab import files; files.download(out_path)   # bỏ # để tải về máy

## 6. (Tùy chọn) Xử lý cả thư mục

In [ ]:
# ==== (TÙY CHỌN) XỬ LÝ CẢ THƯ MỤC ====
# Tự động xóa theo đề xuất cho mọi ảnh trong INPUT_DIR. Lưu ảnh sạch + ảnh đánh dấu + bảng CSV.
import glob, os
INPUT_DIR = "/content/input"     # vd. "/content/drive/MyDrive/anh"  (cần mount Drive ở ô dưới)
# from google.colab import drive; drive.mount("/content/drive")

paths = sorted(p for ext in ("jpg", "jpeg", "png", "webp", "JPG", "JPEG", "PNG")
               for p in glob.glob(os.path.join(INPUT_DIR, f"*.{ext}")))
print(len(paths), "ảnh")
rows = []
for p in paths:
    im = cv2.imread(p)
    if im is None:
        print("bỏ qua:", p); continue
    ob = analyze(im)
    ids_ = [o["id"] for o in ob if o["suggest"]]
    res_, _ = remove_objects(im, ob, ids_)
    stem = os.path.splitext(os.path.basename(p))[0]
    cv2.imwrite(os.path.join(OUTPUT_DIR, stem + "_clean.png"), res_)
    cv2.imwrite(os.path.join(OUTPUT_DIR, stem + "_marked.jpg"), draw_overlay(im, ob))
    t = objs_table(ob); t.insert(0, "ảnh", stem); rows.append(t)
    print(f"{stem}: xóa {len(ids_)} vật")
if rows:
    pd.concat(rows).to_csv(os.path.join(OUTPUT_DIR, "report.csv"), index=False)
    print("Báo cáo:", os.path.join(OUTPUT_DIR, "report.csv"))

## 7. (Tùy chọn) Giao diện app demo — Gradio

In [ ]:
!pip install -q gradio

In [ ]:
# ==== (TÙY CHỌN) GIAO DIỆN APP DEMO (Gradio) ====
# Chạy xong sẽ có link public (share=True) để mở trên điện thoại / gửi người khác thử.
import gradio as gr


def _label(o):
    tag = " [dính chủ thể]" if o["role"] == "attached" else ""
    return f"#{o['id']} {o['name']} ({o['score']:.2f}){tag}"


def ui_analyze(img_rgb):
    if img_rgb is None:
        return None, gr.update(choices=[], value=[]), None, None
    bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
    ob = analyze(bgr)
    cands = [o for o in ob if o["role"] in ("candidate", "attached")]   # vật dính chủ thể vẫn liệt kê, không tick
    vis = cv2.cvtColor(draw_overlay(bgr, ob), cv2.COLOR_BGR2RGB)
    return (vis,
            gr.update(choices=[_label(o) for o in cands], value=[_label(o) for o in cands if o["suggest"]]),
            {"img": bgr, "objs": ob},
            objs_table(ob))


def ui_remove(state, selected):
    if not state:
        return None
    ids_ = [int(s.split()[0][1:]) for s in selected]
    res_, _ = remove_objects(state["img"], state["objs"], ids_)
    return cv2.cvtColor(res_, cv2.COLOR_BGR2RGB)


with gr.Blocks(title="Xóa vật thừa trong ảnh") as demo:
    gr.Markdown("### 1) Tải ảnh → Phân tích  2) Tick/bỏ tick vật muốn xóa → Xóa")
    st = gr.State()
    with gr.Row():
        inp = gr.Image(type="numpy", label="Ảnh gốc")
        vis = gr.Image(label="Đỏ = đề xuất xóa, xanh lá = chủ thể")
    btn_a = gr.Button("Phân tích", variant="primary")
    chk = gr.CheckboxGroup(label="Vật sẽ xóa")
    tbl = gr.Dataframe(label="Chi tiết")
    btn_r = gr.Button("Xóa vật đã chọn", variant="primary")
    out = gr.Image(label="Kết quả")
    btn_a.click(ui_analyze, inp, [vis, chk, st, tbl])
    btn_r.click(ui_remove, [st, chk], out)

demo.launch(share=True, debug=False)